# Aether / Zeno Neural Volume — treinamento remoto opcional

Este notebook usa uma GPU NVIDIA temporária do Google Colab para treinar o protótipo tiny-cuda-nn sem exigir GPU local. No Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU** antes de executar. A disponibilidade e os limites gratuitos dependem da conta e do momento.

Você precisa de um arquivo `.znvds` exportado pelo node `ExportNeuralVolumeDataset` do Zeno. O notebook não inventa dados de uma cena real.

In [ ]:
import torch, os, subprocess, textwrap
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU NVIDIA não alocada. Selecione T4 GPU e reconecte o runtime.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
%cd /content
!git clone --depth 1 --branch zeno-master-clean-copy https://github.com/brucelee1985666-commits/zeno.git zeno
!git clone https://github.com/NVlabs/tiny-cuda-nn.git /content/tiny-cuda-nn
!git -C /content/tiny-cuda-nn checkout 3daa6e5b8b8e0fa17971a1a20acfbb3754494b38
!git -C /content/tiny-cuda-nn submodule update --init --recursive
!pip install -v /content/tiny-cuda-nn/bindings/torch
!pip install 'numpy>=1.24'

## Forneça seu `.znvds`

Você pode montar o Google Drive ou enviar o arquivo pelo seletor de arquivos do Colab. No celular, o upload pode levar algum tempo para volumes grandes.

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione o arquivo .znvds exportado do Zeno
datasets = [name for name in uploaded if name.lower().endswith('.znvds')]
if not datasets:
    raise RuntimeError('Nenhum arquivo .znvds foi enviado.')
DATASET = '/content/' + datasets[0]
OUTPUT = '/content/output.znvm'
print('Dataset:', DATASET, 'bytes:', os.path.getsize(DATASET))

In [ ]:
%cd /content/zeno
!python tools/neural_volume/inspect_znvds.py "$DATASET" --domain-size 128
!python tools/neural_volume/train_znvds.py "$DATASET" "$OUTPUT" --domain-size 128 --steps 2000 --batch-size 32768 --value-tolerance 0.001
!python tools/neural_volume/inspect_znvm.py "$OUTPUT"

In [ ]:
from google.colab import files
files.download(OUTPUT)  # baixe o checkpoint .znvm para seu dispositivo

## Limites importantes

- A GPU do Colab executa treinamento real enquanto a sessão estiver ativa; ela não é uma GPU local nem garante disponibilidade contínua.
- O checkpoint `.znvm` é experimental. O runtime CUDA C++ ainda precisa de validação de inferência em GPU e integração ao ZenoCompute.
- A CI do GitHub valida CPU e compilação CUDA; ela não executa inferência CUDA em hardware.
- Se a sessão gratuita não oferecer GPU ou interromper o trabalho, salve o `.znvds` e tente outra sessão/serviço GPU autorizado.